# Certified Numerical Analysis with Rocq and AI Assistance

If you are using Google Colab, first enable a GPU runtime. Then run the setup cell: it installs the workshop package, downloads the retrieval cache, and creates the retrieval/LLM clients used later.


## 0. Grab a GPU

Google Colab only.

Before doing anything, try to get a GPU instance. A T4 runtime is enough for the local retrieval embeddings used in the notebook.


Go to instance options.

![step_0](https://raw.githubusercontent.com/theostos/integral-tp/refs/heads/main/img/step_0_option.png)

Change runtime.

![step_1](https://raw.githubusercontent.com/theostos/integral-tp/refs/heads/main/img/step_1_change_runtime.png)

Add a T4 GPU.

![step_2](https://raw.githubusercontent.com/theostos/integral-tp/refs/heads/main/img/step_2_t4.png)

Then connect to the GPU instance.

![step_3](https://raw.githubusercontent.com/theostos/integral-tp/refs/heads/main/img/step_3_connect.png)


In [ ]:
%pip install -q integral-tp[colab]@git+https://github.com/theostos/integral-tp.git

import os

from huggingface_hub import hf_hub_download

# Workshop services. Change these values when the servers run elsewhere.
# For one ngrok URL routed by path, set:
# os.environ["ROCQ_SERVER_URL"] = "https://<ngrok-host>/rocq"
# os.environ["WORKSHOP_LLM_SERVER_URL"] = "https://<ngrok-host>/llm"
os.environ.setdefault("ROCQ_SERVER_HOST", "127.0.0.1")
os.environ.setdefault("ROCQ_SERVER_PORT", "5000")
os.environ.setdefault("WORKSHOP_LLM_SERVER_URL", "http://127.0.0.1:8010")
os.environ.setdefault("WORKSHOP_LLM_SERVER_POLL_SECONDS", "2.0")
os.environ.setdefault("MISTRAL_MODEL", "mistral-medium-latest")


import workshop_api
from workshop_api import (
    LLMClient,
    RetrievalClient,
    RetrievalExplorer,
    download_retrieval_cache,
    format_retrieval_hits,
)

zip_path = hf_hub_download(
    repo_id="theostos/integral-tp-retrieval-cache",
    filename="retrieval_cache.zip",
    repo_type="dataset",
)

cache_path = download_retrieval_cache(zip_path, cache_dir="/content/rocq-doc-cache")
retriever = RetrievalClient.from_env(cache_dir=cache_path)
remote_llm = LLMClient.from_env()

_ = retriever.search("warmup", library="Coquelicot", k=3)


## Goal of the session

We start from a badly estimated integral, then use Rocq to certify that the estimate is wrong. After that, we look for an analytic form of the integral by combining three ingredients:

- a retrieval system over Rocq libraries;
- a language model that proposes formal statements and proof scripts;
- Rocq feedback, which accepts correct scripts and rejects incorrect ones.

The point is not to write all proof scripts by hand. The point is to ask: which formal ingredients do we need, where can we find them, and how do we let the model do the boring plumbing while Rocq checks the result?


## 1. A numerical disagreement

We study

$$
 f(x) = \operatorname{sech}(10x-2)^2
      + \operatorname{sech}(100x-40)^4
      + \operatorname{sech}(1000x-600)^6.
$$

A direct Mathematica call such as

```Mathematica
Clear[x, f];
f[x_] := Sech[10 x - 2]^2 + Sech[100 x - 40]^4 + Sech[1000 x - 600]^6;
NIntegrate[f[x], {x, 0, 1}]
```

prints about `0.209736`.

Sage can even return `nan` with the unstable exponential definition of `sech`. With a numerically stable definition it agrees with Mathematica:

```python
var('x')

def sech_stable(u):
    return 2*exp(-abs(u))/(1 + exp(-2*abs(u)))

f = sech_stable(10*x - 2)^2 \
  + sech_stable(100*x - 40)^4 \
  + sech_stable(1000*x - 600)^6

I_num, err = numerical_integral(f, 0, 1)
print(I_num, err)
```

which gives about `0.2097360688339336`.

Let us ask Rocq for a certified enclosure instead of trusting floating point code.


We use a first Rocq document only for the certified numerical part. Later we will create a second document for the analytic proof. This is intentional: if we reset or replay a proof while experimenting with the LLM, we do not want to trigger the expensive interval computation again.


In [ ]:
doc_numerical_integral = workshop_api.new_document()

doc_numerical_integral.add_import("Coq", "Reals Lra Psatz")
doc_numerical_integral.add_import("Coquelicot", "Coquelicot")
doc_numerical_integral.add_import("Interval", "Tactic Plot")


In [ ]:
doc_numerical_integral.add_definition("""Definition sech (u : R) : R :=
  2 * exp (u) / (exp (2 * u) + 1).""")

doc_numerical_integral.add_definition("""Definition f (x : R) : R :=
    (sech (10 * x - 2))^2
  + (sech (100 * x - 40))^4
  + (sech (1000 * x - 600))^6.""")

doc_numerical_integral.add_definition("Definition I : R := RInt f 0 1.")


The command below asks Interval to compute a certified decimal enclosure. It is the only heavy numerical cell in the notebook.


In [ ]:
doc_numerical_integral.execute("""Do integral
  ltac:(let J := eval cbv [I f sech] in I in exact J)
  with (i_prec 25, i_degree 3, i_fuel 300,
        i_width (-15), i_decimal).""")


On the reference file this prints:

```text
0.21078887 <= I <= 0.21081871
```

This is incompatible with the value printed by Mathematica/Sage. We can package the statement as a theorem. This is the only proof in the session that we run manually from start to finish.


In [ ]:
I_digits = doc_numerical_integral.add_theorem("""Theorem I_first_4_decimal_digits :
  Rabs (I - 0.2108) <= 1e-4.""")

I_digits.run_tac("unfold I, f, sech.")
I_digits.run_tac("integral with (i_prec 25, i_degree 3, i_fuel 300).")
I_digits.qed()


## 2. Rocq objects we will manipulate

A quick vocabulary check:

- A **goal** is what remains to prove.
- A **lemma** or **theorem** is a named formal statement. Once proved, it can be reused later.
- A **tactic** is a command that transforms the current goals. Some tactics solve goals; others split them into simpler goals.
- Retrieval can return both lemmas and tactics. Both are useful: a tactic may compute a derivative, while a lemma may justify a side condition.

For the analytic part, the workflow will be:

1. State what mathematical/formal ingredient we need, without guessing its Rocq name.
2. Use the retrieval explorer to collect relevant library facts or tactics.
3. Send the current proof state plus the selected context to the LLM.
4. Let Rocq check the proposed proof script.


## 3. Analytic document

We now start a fresh Rocq document. It repeats the definitions, but it does not run the interval computation.


In [ ]:
doc_analytic_integral = workshop_api.new_document()

doc_analytic_integral.add_import("Coq", "Reals Lra Psatz")
doc_analytic_integral.add_import("Coquelicot", "Coquelicot")
doc_analytic_integral.add_import("Interval", "Tactic Plot")


In [ ]:
doc_analytic_integral.add_definition("""Definition sech (u : R) : R :=
  2 * exp (u) / (exp (2 * u) + 1).""")

doc_analytic_integral.add_definition("""Definition f (x : R) : R :=
    (sech (10 * x - 2))^2
  + (sech (100 * x - 40))^4
  + (sech (1000 * x - 600))^6.""")

doc_analytic_integral.add_definition("Definition I : R := RInt f 0 1.")


A language model suggests the usual antiderivatives of even powers of `sech`. We use the exponential presentation of the hyperbolic tangent,

$$
\tanh_{\exp}(u) = \frac{e^{2u} - 1}{e^{2u} + 1}.
$$

This avoids the Stdlib name `tanh`, which is already defined through `sinh` and `cosh`.


In [ ]:
doc_analytic_integral.add_definition("""Definition tanh_exp (u : R) : R :=
  (exp (2 * u) - 1) / (exp (2 * u) + 1).""")

doc_analytic_integral.add_definition("""Definition A2 (u : R) : R :=
  tanh_exp u.""")

doc_analytic_integral.add_definition("""Definition A4 (u : R) : R :=
  tanh_exp u - (/ 3) * (tanh_exp u)^3.""")

doc_analytic_integral.add_definition("""Definition A6 (u : R) : R :=
  tanh_exp u - (2 / 3) * (tanh_exp u)^3 + (/ 5) * (tanh_exp u)^5.""")


We rescale these antiderivatives to match the three bumps of the integrand.


In [ ]:
doc_analytic_integral.add_definition("""Definition F2 (x : R) : R :=
  A2 (10 * x - 2) / 10.""")

doc_analytic_integral.add_definition("""Definition F4 (x : R) : R :=
  A4 (100 * x - 40) / 100.""")

doc_analytic_integral.add_definition("""Definition F6 (x : R) : R :=
  A6 (1000 * x - 600) / 1000.""")

doc_analytic_integral.add_definition("""Definition F (x : R) : R :=
  F2 x + F4 x + F6 x.""")

doc_analytic_integral.add_definition("""Definition I_closed_form : R :=
  F 1 - F 0.""")


Conceptually, we want to prove:

- `F2' x = sech(10*x - 2)^2`
- `F4' x = sech(100*x - 40)^4`
- `F6' x = sech(1000*x - 600)^6`

Then `F = F2 + F4 + F6` is an antiderivative of `f`, so the fundamental theorem of calculus gives `I = F 1 - F 0`.


In [ ]:
# This list is the context that we curate during retrieval.
# RetrievalExplorer writes into it, and we can also add already proved objects.
selected_hits = []


## 4. Formalizing `F2_derivative`

First we need to express the informal statement "the derivative of `F2` is `sech(10*x - 2)^2`" in Coquelicot's language.

Search for the predicate used to say that a real function has a derivative at a point.

<details>
<summary>Hint</summary>

Try a query like: `real function has derivative at a point predicate Coquelicot`.

</details>


In [ ]:
selected_hits.clear()

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_k=8,
)
retrieval_explorer.display()


Based on the search results, did you find a definition that can express the derivative of `F2`?

<details>
<summary>Expected retrieval hit</summary>

Name: `is_derive`

Signature:

```coq
is_derive : (R -> R) -> R -> R -> Prop
```

</details>

Now fill the following cell with the formal statement.


In [ ]:
f2_derivative = doc_analytic_integral.add_theorem("""Lemma F2_derivative (x : R) :
  [TO FILL].""")

<details>
<summary>Expected answer</summary>

```coq
Lemma F2_derivative (x : R) :
  is_derive F2 x ((sech (10 * x - 2)) ^ 2).
```

</details>


To prove this, we need to show two things:

- `F2` is differentiable at `x`;
- the derivative computed from the definition of `F2` is equal to the right-hand side.

So we are looking for a tactic that helps with differentiability and computes derivatives of real functions.

<details>
<summary>Hint</summary>

Try a query like: `automatic tactic compute derivative differentiability real functions Coquelicot`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `auto_derive`

Signature:

```coq
Ltac auto_derive := ...
```

It is a Coquelicot tactic for derivative and differentiability goals.

</details>


In [ ]:
selected_hits.clear()

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="Ltac",
    default_k=8,
)
retrieval_explorer.display()


Now we run only the first proof steps by hand. This is useful: `unfold` exposes the definitions, and the derivative tactic exposes the side conditions that are really needed.


In [ ]:
f2_derivative.run_tac("unfold F2, A2, sech, tanh_exp.")
f2_derivative.run_tac("auto_derive.")
print(f2_derivative)

# here we do a checkpoint so that later we will be able to
# reset the proof to this more advance point
f2_derivative.checkpoint("after_auto_derive")


Inspect the goals. One of them asks us to prove that an expression of the form `exp u + 1` is not zero.

This should not be solved only locally: the same denominator appears in `F4`, `F6`, and the differentiability proof for `f`. We therefore create an independent lemma.

Formal ingredients we need:

- the exponential is strictly positive;
- the sum of two strictly positive real numbers is strictly positive;
- a strictly positive real number is not zero;
- a linear arithmetic tactic can finish facts such as `1 > 0`.

<details>
<summary>Hint</summary>

Try a query like: `exponential positive sum positive real positive not zero real`.

</details>

<details>
<summary>Expected retrieval hits</summary>

Name: `exp_pos`

Signature:

```coq
exp_pos : forall x : R, 0 < exp x
```

Name: `Rplus_lt_0_compat`

Signature:

```coq
Rplus_lt_0_compat : forall r1 r2 : R, 0 < r1 -> 0 < r2 -> 0 < r1 + r2
```

Name: `Rgt_not_eq`

Signature:

```coq
Rgt_not_eq : forall r1 r2 : R, r1 > r2 -> r1 <> r2
```

</details>


In [ ]:
selected_hits.clear()

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Stdlib",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


In [ ]:
sech_denominator_nonzero = doc_analytic_integral.add_theorem("""Lemma sech_denominator_nonzero (u : R) :
  exp u + 1 <> 0.""")

result = remote_llm.prove(
    sech_denominator_nonzero,
    selected_hits=selected_hits,
    extra_context=(
        "denominator is nonzero because it is strictly "
        "positive: the exponential is positive, and adding 1 preserves "
        "strict positivity."
    ),
    verbose=True,
    tools={
        "run_tac": sech_denominator_nonzero.run_tac,
        "reverse": sech_denominator_nonzero.reverse,
    },
    max_tool_calls=30,
)
print(result)


<details>
<summary>Expected proof</summary>

```coq
apply Rgt_not_eq with (r1 := ((exp u) + 1)) (r2 := 0).
apply Rplus_lt_0_compat with (r1 := exp u) (r2 := 1).
apply exp_pos.
lra.
```

</details>


The lemma is now a local result we proved during the session. We add it to the selected context so the LLM can use it exactly like a library fact.

There is one more technical ingredient in the derivative proofs. After `auto_derive`, the equality goal contains expressions such as `exp (2 * u)`, while `sech` contains `exp u`. We need the theorem saying that the exponential of a sum is a product. The standard algebra tactics `ring`, `field`, and `nra` are already available; `exp_plus` is the specific reusable lemma to gather here.

<details>
<summary>Hint</summary>

Try a query like: `exponential of a sum is product exp x plus y`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `exp_plus`

Signature:

```coq
exp_plus : forall x y : R, exp (x + y) = exp x * exp y
```

</details>


In [ ]:
selected_hits.clear()
selected_hits.append(sech_denominator_nonzero.as_retrieval_hit())

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Stdlib",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


After adding `exp_plus`, save this small technical context. It will be reused for `F2`, `F4`, and `F6`.


In [ ]:
technical_derivative_hits = list(selected_hits)
print(format_retrieval_hits(technical_derivative_hits))


## 5. Three LLM strategies on `F2_derivative`

We now try three ways of asking the LLM to finish the same proof state. Before each experiment we roll back to the checkpoint just after `auto_derive`, so all strategies start from the same goals.

The key ingredients in the selected context should now be:

- `sech_denominator_nonzero`, for the denominator side condition;
- `exp_plus`, for rewriting `exp (u + u)` into `exp u * exp u` in the equality goal.

Run one strategy first. If you want to compare strategies, roll back to `after_auto_derive` between attempts.

Strategy A: one direct call to `prove`. The notebook builds the prompt; participants do not need to edit it.


In [ ]:
f2_max_attempts = 10
result_direct = None

for attempt_id in range(1, f2_max_attempts + 1):
    f2_derivative.reverse("after_auto_derive")
    print(f"Direct attempt {attempt_id}/{f2_max_attempts}")
    result_direct = remote_llm.prove(
        f2_derivative,
        selected_hits=selected_hits,
        verbose=True,
        close=False,
    )
    print(result_direct)
    if result_direct.ok:
        break

assert result_direct is not None and result_direct.ok
f2_derivative.reverse("after_auto_derive")


Strategy B: feedback loop. We do not need a special API method for this. We call `prove`; if the proof does not work, we call `prove` again with the previous attempt and the Rocq error in `extra_context`.


In [ ]:
f2_derivative.reverse("after_auto_derive")


def result_ok(result):
    if isinstance(result, dict):
        return bool(result.get("ok"))
    return bool(getattr(result, "ok", False))


def result_part(result, name):
    if isinstance(result, dict):
        return result.get(name, "")
    return getattr(result, name, "")


feedback_context = ""
result_feedback = None

for round_id in range(3):
    f2_derivative.reverse("after_auto_derive")
    print(f"Feedback attempt {round_id}/2")
    result_feedback = remote_llm.prove(
        f2_derivative,
        selected_hits=selected_hits,
        extra_context=feedback_context,
        verbose=True,
        close=False,
    )
    print(result_feedback)

    if result_ok(result_feedback):
        break

    feedback_context += f"""
Previous attempt #{round_id + 1}:
{result_part(result_feedback, "script")}

Rocq feedback:
{result_part(result_feedback, "error")}
"""

assert result_feedback is not None and result_ok(result_feedback)
f2_derivative.reverse("after_auto_derive")


Strategy C: agentic loop. This is still the same `prove` method, but we give it tools. The model can apply a tactic, inspect the new proof state, and roll back with `reverse`. The retrieval step stays outside the agent loop: we explicitly give the model the selected ingredients.


In [ ]:
f2_derivative.reverse("after_auto_derive")

result_agentic = remote_llm.prove(
    f2_derivative,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f2_derivative.run_tac,
        "reverse": f2_derivative.reverse,
    },
    max_tool_calls=48
)
print(result_agentic)


After one strategy succeeds, add the proved derivative lemma to the context. We will reuse it when proving that the sum `F` is an antiderivative of `f`.


In [ ]:
selected_hits.append(f2_derivative.as_retrieval_hit())
print(format_retrieval_hits(selected_hits))


## 6. `F4_derivative`

The formal task is the same, but the right-hand side is the fourth power. The useful ingredients are:

- a tactic that computes derivatives;
- the denominator lemma we already proved;
- `exp_plus`, to rewrite the exponential of a sum in the equality goal;
- optionally, the complete proof of `F2_derivative` as an example of a very similar successful proof.

As for `F2_derivative`, we first run the transparent steps by hand: unfold the definitions, then run the derivative tactic. After that, participants can use any strategy from Section 5: direct `prove`, the feedback loop, or `prove` with tools.

<details>
<summary>Hint</summary>

Try a query like: `automatic tactic compute derivative differentiability real functions Coquelicot`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `auto_derive`

Signature:

```coq
Ltac auto_derive := ...
```

</details>


In [ ]:
f4_derivative = doc_analytic_integral.add_theorem("""Lemma F4_derivative (x : R) :
  is_derive F4 x ((sech (100 * x - 40)) ^ 4).""")

selected_hits.clear()
selected_hits.extend(technical_derivative_hits)
selected_hits.append(f2_derivative.as_retrieval_hit())

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="Ltac",
    default_k=8,
)
retrieval_explorer.display()


In [ ]:
f4_derivative.run_tac("unfold F4, A4, sech, tanh_exp.")
f4_derivative.run_tac("auto_derive.")
print(f4_derivative)

f4_derivative.checkpoint("after_auto_derive")


The cell below uses the agentic strategy again. The selected context contains the reusable denominator lemma, the exponential addition theorem, and the successful `F2_derivative` proof as an example. The extra context only describes the mathematical analogy with the previous derivative.


In [ ]:
f4_derivative.reverse("after_auto_derive")

result = remote_llm.prove(
    f4_derivative,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f4_derivative.run_tac,
        "reverse": f4_derivative.reverse,
    },
    max_tool_calls=48,
    extra_context="""""",
)
print(result)


In [ ]:
selected_hits.append(f4_derivative.as_retrieval_hit())


## 7. `F6_derivative`

The sixth-power proof is again the same pattern. The formal ingredients are the derivative-computation tactic, the reusable denominator lemma, and `exp_plus` for the equality goal. There are more denominator side conditions, but conceptually nothing new.

Again, we first run `unfold` and `auto_derive` manually so the proof state is visible. After that, participants can reuse any of the strategies from Section 5.


In [ ]:
f6_derivative = doc_analytic_integral.add_theorem("""Lemma F6_derivative (x : R) :
  is_derive F6 x ((sech (1000 * x - 600)) ^ 6).""")

selected_hits.clear()
selected_hits.extend(technical_derivative_hits)
selected_hits.extend([f2_derivative.as_retrieval_hit(), f4_derivative.as_retrieval_hit()])

f6_derivative.run_tac("unfold F6, A6, sech, tanh_exp.")
f6_derivative.run_tac("auto_derive.")
print(f6_derivative)

f6_derivative.checkpoint("after_auto_derive")

result = remote_llm.prove(
    f6_derivative,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f6_derivative.run_tac,
        "reverse": f6_derivative.reverse,
    },
    max_tool_calls=48,
    extra_context="""""",
)
print(result)


In [ ]:
selected_hits.append(f6_derivative.as_retrieval_hit())

## 8. Combining the derivative lemmas

Now we need to prove that the derivative of the sum `F2 + F4 + F6` is the sum of the derivatives. The formal ingredient to retrieve is the Coquelicot rule for the derivative of a sum.

<details>
<summary>Hint</summary>

Try a query like: `derivative of the sum of two real functions`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `is_derive_plus`

Signature:

```coq
is_derive_plus :
  forall (f g : R -> R) (x df dg : R),
    is_derive f x df ->
    is_derive g x dg ->
    is_derive (fun y => f y + g y) x (df + dg)
```

</details>


In [ ]:
selected_hits.clear()
selected_hits.extend([
    f2_derivative.as_retrieval_hit(),
    f4_derivative.as_retrieval_hit(),
    f6_derivative.as_retrieval_hit(),
])

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


In [ ]:
f_derivative = doc_analytic_integral.add_theorem("""Lemma F_derivative (x : R) :
  is_derive F x (f x).""")

result = remote_llm.prove(
    f_derivative,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f_derivative.run_tac,
        "reverse": f_derivative.reverse,
    },
    max_tool_calls=40,
    extra_context="""Use the derivative rule for sums together with the derivative lemmas already proved.""",
)
print(result)


## 9. What the integral theorem asks for

We now want to prove the closed form of the integral. Mathematically, the plan is to show that `F` is an antiderivative of `f` on the interval, then use the fundamental theorem of calculus to turn the integral into `F 1 - F 0`.

There is a small formal distinction to keep in mind. The theorem of calculus usually gives a predicate of the form `is_RInt f 0 1 v`, meaning that `v` is the value of the integral. But our definition uses `I := RInt f 0 1`, a real number. So we need two different ingredients:

- one theorem that turns an antiderivative plus continuity into an `is_RInt` statement;
- one uniqueness bridge that turns `is_RInt f 0 1 v` into the equality `RInt f 0 1 = v`.

These are conceptually different, so we search for them separately.

<details>
<summary>Hint for the fundamental theorem step</summary>

Try a query like: `fundamental theorem calculus real integral derivative interval is_RInt`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `is_RInt_derive`

Signature:

```coq
is_RInt_derive :
  forall (f g : R -> R) (a b : R),
    (forall x, Rmin a b <= x <= Rmax a b -> is_derive f x (g x)) ->
    (forall x, Rmin a b <= x <= Rmax a b -> continuous g x) ->
    is_RInt g a b (f b - f a)
```

</details>


In [ ]:
integral_hits = []

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=integral_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


The previous theorem gives a predicate statement. Since our goal is an equality about `RInt`, we also need the uniqueness bridge from the predicate form to the function value.

<details>
<summary>Hint for the uniqueness bridge</summary>

Try a query like: `RInt value unique is_RInt equality`.

</details>

<details>
<summary>Expected retrieval hit</summary>

Name: `is_RInt_unique`

Signature:

```coq
is_RInt_unique : forall f a b If, is_RInt f a b If -> RInt f a b = If
```

</details>


In [ ]:
retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=integral_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


Inspecting these statements reveals two obligations for the final theorem. The antiderivative obligation is already handled by `F_derivative`. The remaining missing ingredient is continuity of `f` on the interval.

We prove that continuity fact through two small technical lemmas: first show that `f` is differentiable, then use the general theorem saying that differentiability implies continuity.

## 10. The missing continuity lemmas

For the differentiability lemma, use the denominator lemma and the derivative-computation tactic. The formal strategy is to unfold both the outer definition `f` and the helper definition `sech` before asking the derivative tactic to finish the structural work.

<details>
<summary>Hint for the differentiability lemma</summary>

Try a query like: `automatic tactic prove derivative exists real function Coquelicot`.

</details>

<details>
<summary>Expected retrieval hit for the differentiability lemma</summary>

Name: `auto_derive`

Signature:

```coq
Ltac auto_derive := ...
```

</details>


In [ ]:
selected_hits.clear()
selected_hits.append(sech_denominator_nonzero.as_retrieval_hit())

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="Ltac",
    default_k=8,
)
retrieval_explorer.display()

In [ ]:
f_ex_derive = doc_analytic_integral.add_theorem("""Lemma f_ex_derive (x : R) :
  ex_derive f x.""")

result = remote_llm.prove(
    f_ex_derive,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f_ex_derive.run_tac,
        "reverse": f_ex_derive.reverse,
    },
    max_tool_calls=36,
)
print(result)


Now we turn differentiability into continuity. The formal ingredient is the general theorem saying that a differentiable real function is continuous at the point.

<details>
<summary>Hint for the continuity lemma</summary>

Try a query like: `differentiable real function is continuous`.

</details>

<details>
<summary>Expected retrieval hit for the continuity lemma</summary>

Name: `ex_derive_continuous`

Signature:

```coq
ex_derive_continuous :
  forall (f : R -> R) (x : R), ex_derive f x -> continuous f x
```

</details>


In [ ]:
selected_hits.clear()
selected_hits.append(f_ex_derive.as_retrieval_hit())

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="start_theorem_proof",
    default_k=8,
)
retrieval_explorer.display()


In [ ]:
f_continuous = doc_analytic_integral.add_theorem("""Lemma f_continuous (x : R) :
  continuous f x.""")

result = remote_llm.prove(
    f_continuous,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f_continuous.run_tac,
        "reverse": f_continuous.reverse,
    },
    max_tool_calls=16,
)
print(result)


## 11. Closed form of the integral

We can now return to the theorem that motivated the continuity detour. The final context contains:

- the integration facts collected in Section 9;
- the antiderivative fact `F_derivative`;
- the continuity fact `f_continuous`.


In [ ]:
selected_hits.clear()
selected_hits.extend(integral_hits)
selected_hits.extend([
    f_derivative.as_retrieval_hit(),
    f_continuous.as_retrieval_hit(),
])

[hit.get("name") for hit in selected_hits]


In [ ]:
I_closed_form_correct = doc_analytic_integral.add_theorem("""Theorem I_closed_form_correct :
  I = I_closed_form.""")

result = remote_llm.prove(
    I_closed_form_correct,
    selected_hits=selected_hits,
    extra_context=(
        "Since F is an antiderivative of f on the interval and f is continuous, "
        "the integral of f from 0 to 1 is F(1) - F(0)."
    ),
    verbose=True,
    tools={
        "run_tac": I_closed_form_correct.run_tac,
        "reverse": I_closed_form_correct.reverse,
    },
    max_tool_calls=24,
)
print(result)


The formal result is now:

```coq
I = F 1 - F 0
```

So the numerical dispute can be checked again by evaluating a stable version of the closed form in ordinary Python. This final Python calculation is not the proof; the proof is the Rocq theorem above.


In [ ]:
import math


def A2_num(u):
    t = math.tanh(u)
    return t


def A4_num(u):
    t = math.tanh(u)
    return t - (1 / 3) * t**3


def A6_num(u):
    t = math.tanh(u)
    return t - (2 / 3) * t**3 + (1 / 5) * t**5


def F_num(x):
    return (
        A2_num(10 * x - 2) / 10
        + A4_num(100 * x - 40) / 100
        + A6_num(1000 * x - 600) / 1000
    )

F_num(1) - F_num(0)


The value is about `0.2108027355005493`, which lies inside the certified Rocq interval and explains why the earlier `0.209736...` estimate was wrong.


## 12. Source generated by the analytic document

For debugging or comparison with `integral.v`, print the source accumulated in the analytic document.


In [ ]:
print(doc_analytic_integral.source())
